# dataset

In [8]:
import os
import shutil
import random

# 原始数据路径
data_root = "tracked_images"
dataset_root = "dataset_nondrown_10_percent"

# 目标数据集路径
train_dir = os.path.join(dataset_root, "train_data")
val_dir = os.path.join(dataset_root, "val_data")

# 目标类别
categories = ["drown", "non-drown"]

# 确保目录结构存在
for split in [train_dir, val_dir]:
    for category in categories:
        os.makedirs(os.path.join(split, category), exist_ok=True)

# 获取所有视频编号
all_videos = sorted(os.listdir(data_root))

# 手动指定的验证集视频
val_videos = {"video_66", "video_44", "video_51", "video_21", "video_70"}

# 剩余的视频，计算额外的 20% 作为验证集
remaining_videos = list(set(all_videos) - val_videos)
extra_val_count = int(len(remaining_videos) * 0.2)
random.seed(42)  
extra_val_videos = set(random.sample(remaining_videos, extra_val_count))
final_val_videos = val_videos.union(extra_val_videos)

print(f"最终验证集视频数量: {len(final_val_videos)}")
print(f"验证集视频: {final_val_videos}")

# 遍历所有视频
for video in all_videos:
    video_path = os.path.join(data_root, video)

    for person in os.listdir(video_path):
        person_path = os.path.join(video_path, person)

        # 确定当前 person 是 drown 还是 non-drown
        label = "drown" if "(drown)" in person else "non-drown"

        # 获取所有图片
        all_images = sorted(os.listdir(person_path))

        # 目标路径
        target_dir = val_dir if video in final_val_videos else train_dir
        target_label_dir = os.path.join(target_dir, label)

        # 计算需要保留的图片
        if label == "drown":
            # drown 类别的图片全部保留
            selected_images = all_images
        else:
            # non-drown 采样 10%，但至少保留 2 张
            min_keep = 2
            num_to_keep = max(min_keep, int(len(all_images) * 0.1))
            selected_images = random.sample(all_images, num_to_keep) if len(all_images) > num_to_keep else all_images

        for img_file in selected_images:
            src_path = os.path.join(person_path, img_file)

            # 构造新的文件名：video_X_person_Y_原始图片名
            new_filename = f"{video}_{person}_{img_file}"
            dest_path = os.path.join(target_label_dir, new_filename)

            # 复制文件到新路径
            shutil.copy(src_path, dest_path)

print(f"数据整理完成！")
print(f"训练数据路径: {train_dir}")
print(f"验证数据路径: {val_dir}")
print(f"最终验证集视频数量: {len(final_val_videos)}")


最终验证集视频数量: 16
验证集视频: {'video_15', 'video_70', 'video_68', 'video_63', 'video_53', 'video_21', 'video_43', 'video_11', 'video_54', 'video_6', 'video_22', 'video_28', 'video_60', 'video_51', 'video_66', 'video_44'}
数据整理完成！
训练数据路径: dataset_nondrown_10_percent\train_data
验证数据路径: dataset_nondrown_10_percent\val_data
最终验证集视频数量: 16


In [9]:
import shutil

zip_filename = "dataset_nondrown_10_percent.zip"
shutil.make_archive(dataset_root, 'zip', dataset_root)
print(f"数据压缩完成！压缩文件: {zip_filename}")


数据压缩完成！压缩文件: dataset_nondrown_10_percent.zip


# 解压上传的zip文件

In [ ]:
import zipfile
import os
from tqdm import tqdm
from google.colab import drive

# 挂载 Google Drive
drive.mount('/content/drive')

# 目标解压目录
extract_path = "dataset"
zip_path = "/content/drive/My Drive/dataset_nondrown_10_percent.zip"

# 获取 ZIP 文件的总大小
total_size = os.path.getsize(zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_list = zip_ref.infolist()  
    extracted_size = 0  

    with tqdm(total=total_size, desc="解压进度", unit="B", unit_scale=True) as pbar:
        for file in file_list:
            zip_ref.extract(file.filename, extract_path)
            extracted_size += file.file_size
            pbar.update(file.file_size) 

print(f"✅ 数据集已解压到: {extract_path}")


# EfficientNet_B3训练

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
import numpy as np

# ================== 配置 ================== #
DATASET_ROOT = "dataset"
TRAIN_DIR = f"{DATASET_ROOT}/train_data"
VAL_DIR = f"{DATASET_ROOT}/val_data"
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4
MODEL_PATH = "best_drown_model_EfficientNet_B3.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================== 数据增强 ================== #
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2), shear=10),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ================== 数据加载 ================== #
train_dataset = ImageFolder(root=TRAIN_DIR, transform=train_transform)
val_dataset   = ImageFolder(root=VAL_DIR,   transform=val_transform)

# 翻转 drown=1, non-drown=0
train_dataset.samples = [(path, 1 - label) for path, label in train_dataset.samples]
val_dataset.samples   = [(path,   1 - label) for path, label in val_dataset.samples]

train_dataset.targets = [label for _, label in train_dataset.samples]
val_dataset.targets   = [label for _, label in val_dataset.samples]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# ================== 计算类别权重 ================== #
labels = train_dataset.targets
class_counts = np.bincount(labels)  
class_weights = 1.0 / class_counts  # 计算类别权重
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE) 

# ================== 定义模型 ================== #
def build_model(num_classes=2):
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)  # 使用 EfficientNet-B3 预训练权重
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)  # 修改分类层
    nn.init.xavier_uniform_(model.classifier[1].weight) 
    return model.to(DEVICE)

model = build_model()

# ================== 优化器 & 学习率调度 ================== #
criterion = nn.CrossEntropyLoss(weight=class_weights)  # 加入类别权重
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ================== 训练函数 ================== #
def train_one_epoch(model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)  
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    train_precision = precision_score(all_labels, all_preds, average="binary", pos_label=1)
    train_recall    = recall_score(all_labels, all_preds, average="binary",   pos_label=1)
    train_f1        = f1_score(all_labels, all_preds, average="binary",       pos_label=1)

    epoch_loss = running_loss
    epoch_acc  = correct / total if total > 0 else 0

    return epoch_loss, epoch_acc, train_f1, train_precision, train_recall

# ================== 验证函数 ================== #
def evaluate(model, val_loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_precision = precision_score(all_labels, all_preds, average="binary", pos_label=1)
    val_recall    = recall_score(all_labels, all_preds, average="binary",   pos_label=1)
    val_f1        = f1_score(all_labels, all_preds, average="binary",       pos_label=1)

    return running_loss, val_f1, val_precision, val_recall

# ================== 训练过程 ================== #
def train_model(model, train_loader, val_loader, epochs=10, save_path="best_drown_model.pth"):
    best_f1 = 0.0

    for epoch in range(epochs):
        print(f"=== Epoch [{epoch+1}/{epochs}] ===")
        train_loss, train_acc, train_f1, train_precision, train_recall = train_one_epoch(
            model, train_loader, criterion, optimizer
        )
        val_loss, val_f1, val_precision, val_recall = evaluate(model, val_loader, criterion)

        print(f"[Train] Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
              f"F1(drown=1): {train_f1:.4f} | Precision(drown=1): {train_precision:.4f} | Recall(drown=1): {train_recall:.4f}")
        print(f"[Val]   Loss: {val_loss:.4f} | F1(drown=1): {val_f1:.4f} | "
              f"Precision(drown=1): {val_precision:.4f} | Recall(drown=1): {val_recall:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)
            print(f"Best model updated (Val F1={best_f1:.4f}) -> {save_path}")

        scheduler.step()

# ================== 开始训练 ================== #
train_model(model, train_loader, val_loader, epochs=EPOCHS, save_path=MODEL_PATH)


In [17]:
import torch
import torchvision.transforms as transforms
from PIL import Image
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

# 设备
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 载入模型
def load_model(model_path="model/best_drown_model_EfficientNet_B3.pth"):
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(in_features, 2)  # 2分类
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()  # 设为评估模式
    return model

# 加载模型
model = load_model("model/best_drown_model_EfficientNet_B3.pth")

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_drowning(image_path, model):
    image = Image.open(image_path).convert("RGB")  
    image = transform(image).unsqueeze(0).to(DEVICE)  

    with torch.no_grad():
        output = model(image)  
        probs = torch.softmax(output, dim=1)  
        drown_prob = probs[0, 1].item()  
        non_drown_prob = probs[0, 0].item()  

    print(f"Image: {image_path}")
    print(f"Drown Probability: {drown_prob:.4f}")
    print(f"Non-Drown Probability: {non_drown_prob:.4f}")
    print(f"Prediction: {'Drowning' if drown_prob > 0.5 else 'Not Drowning'}")

# 示例：预测多张图片
image_paths = ["test1.png","test2.png","test3.png","test4.png"]
for img_path in image_paths:
    predict_drowning(img_path, model)


Image: test1.png
Drown Probability: 0.0000
Non-Drown Probability: 1.0000
Prediction: Not Drowning
Image: test2.png
Drown Probability: 0.0000
Non-Drown Probability: 1.0000
Prediction: Not Drowning
Image: test3.png
Drown Probability: 0.9199
Non-Drown Probability: 0.0801
Prediction: Drowning
Image: test4.png
Drown Probability: 0.9973
Non-Drown Probability: 0.0027
Prediction: Drowning
